# D103 — Context Managers and the `with` Statement

This notebook explains how Python context managers acquire and release resources safely.

## Learning goals

- Understand what a context manager is.
- Understand how `with` uses `__enter__()` and `__exit__()`.
- See why files are closed even when an exception occurs.
- Implement custom class-based context managers.
- Implement a generator-based context manager with `contextlib.contextmanager`.
- Understand exception propagation and suppression.
- Know which resources benefit from context management.

## 1. Is `with` associated with context managers?

**Yes.** The Python `with` statement is the language syntax for using a context manager.

A synchronous context manager follows this protocol:

- `__enter__()` — acquire or prepare a resource and optionally return an object.
- `__exit__(exc_type, exc_value, traceback)` — release or clean up the resource.

The expression following `with` must produce a context-manager object. Otherwise Python raises `TypeError`.

```python
with some_context_manager as value:
    use(value)
```

`value` receives whatever `__enter__()` returns. Python calls `__exit__()` when control leaves the block, whether it leaves normally, through `return`, through `break`, or because of an exception.

## 2. Why context managers?

Programs frequently acquire something that must later be released:

- Open files must be closed.
- Database transactions must be committed or rolled back.
- Locks must be released.
- Network connections and sessions must be closed.
- Temporary resources may need cleanup.
- Application state may need to be restored.

Manual cleanup is easy to forget, especially when an exception interrupts normal execution. A context manager places acquisition and guaranteed cleanup around a clearly visible block.

## 3. A file is a context manager

`open()` returns a file object that implements the context-manager protocol. Its `__enter__()` returns the file object, and its `__exit__()` closes the file.

In [ ]:
from pathlib import Path

example_file = Path("context_manager_orders.txt")

with example_file.open("w", encoding="utf-8") as file:
    print("Inside block - closed?", file.closed)
    file.write("ORD-1,placed\n")
    file.write("ORD-2,shipped\n")

print("Outside block - closed?", file.closed)

The `as file` name remains visible after the block, but the underlying resource is closed. Trying to read or write through that closed file object raises `ValueError`.

In [ ]:
try:
    file.write("ORD-3,packed\n")
except ValueError as error:
    print(type(error).__name__ + ":", error)

## 4. What happens behind `with`?

This code:

```python
with manager_expression as value:
    block()
```

is conceptually similar to:

```python
manager = manager_expression
value = manager.__enter__()
try:
    block()
except BaseException as error:
    suppress = manager.__exit__(
        type(error), error, error.__traceback__
    )
    if not suppress:
        raise
else:
    manager.__exit__(None, None, None)
```

The real language semantics handle some details more carefully, but this model explains the main lifecycle.

Important points:

- If `__enter__()` itself fails, the block never begins and that manager's `__exit__()` is not called.
- Once `__enter__()` succeeds, Python arranges for `__exit__()` when the block is left.
- On normal completion, all three exception arguments are `None`.
- During an exception, `__exit__()` receives its type, value, and traceback.
- A truthy return from `__exit__()` suppresses the exception; a false value lets it propagate.

## 5. Observe the protocol directly

This small manager prints every lifecycle event.

In [ ]:
class ContextTracer:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"ENTER: acquiring {self.name}")
        return f"value supplied by {self.name}"

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print(f"EXIT: releasing {self.name} after normal completion")
        else:
            print(f"EXIT: releasing {self.name} after {exc_type.__name__}: {exc_value}")

        return False  # do not suppress exceptions

with ContextTracer("demo resource") as supplied_value:
    print("BLOCK:", supplied_value)

## 6. Cleanup when an exception occurs

The manager's cleanup runs before the exception continues to the surrounding `try/except`.

In [ ]:
try:
    with ContextTracer("order import"):
        print("BLOCK: starting import")
        raise ValueError("invalid order row")
except ValueError as error:
    print("OUTER CODE caught:", error)

This is why context managers are safer than placing a cleanup statement only at the end of ordinary code: exceptions cannot skip `__exit__()` after entry succeeds.

## 7. Custom file context manager

`OrderFileManager` wraps a real file. It demonstrates acquisition in `__enter__()` and cleanup in `__exit__()`.

A well-behaved manager should release only the resource it owns.

In [ ]:
class OrderFileManager:
    def __init__(self, path, mode="r", encoding="utf-8"):
        self.path = Path(path)
        self.mode = mode
        self.encoding = encoding
        self.file = None

    def __enter__(self):
        print(f"ENTER: opening {self.path}")
        self.file = self.path.open(self.mode, encoding=self.encoding)
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        if self.file is not None and not self.file.closed:
            print(f"EXIT: closing {self.path}")
            self.file.close()

        if exc_type is not None:
            print(f"EXIT: block raised {exc_type.__name__}: {exc_value}")

        return False

In [ ]:
order_file = Path("custom_manager_orders.txt")

with OrderFileManager(order_file, "w") as output:
    output.write("ORD-101,paid\n")
    output.write("ORD-102,packed\n")
    print("BLOCK: writing; closed?", output.closed)

print("AFTER: closed?", output.closed)

with OrderFileManager(order_file, "r") as source:
    for line in source:
        print("READ:", line.strip())

### The same manager with a failing block

In [ ]:
try:
    with OrderFileManager(order_file, "r") as source:
        print("First line:", source.readline().strip())
        raise RuntimeError("consumer failed while processing")
except RuntimeError as error:
    print("OUTER CODE caught:", error)

print("File still closed after failure?", source.closed)

## 8. Exception suppression

`__exit__()` can return `True` to tell Python that it handled an exception. Suppression should be narrow and intentional; silently swallowing unexpected errors makes programs difficult to debug.

In [ ]:
class SuppressInvalidOrder:
    def __enter__(self):
        print("Ready to validate one order")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        should_suppress = exc_type is ValueError

        if should_suppress:
            print("Handled expected invalid-order error:", exc_value)

        return should_suppress

with SuppressInvalidOrder():
    raise ValueError("quantity cannot be negative")

print("Program continues because that ValueError was suppressed")

The manager suppresses exactly `ValueError`, not every possible exception. Most resource managers return `False` or `None` so application failures remain visible.

## 9. A transaction-style context manager

Context management is not limited to files. The following teaching example models a small transaction:

- Normal completion → commit.
- Exception → rollback.
- Always → close the transaction/session resource.

In [ ]:
class DemoOrderTransaction:
    def __init__(self):
        self.pending_operations = []
        self.closed = False

    def __enter__(self):
        print("BEGIN TRANSACTION")
        return self

    def add(self, operation):
        if self.closed:
            raise RuntimeError("transaction is closed")
        print("STAGE:", operation)
        self.pending_operations.append(operation)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("COMMIT:", self.pending_operations)
        else:
            print("ROLLBACK:", self.pending_operations)
            self.pending_operations.clear()

        self.closed = True
        print("CLOSE TRANSACTION")
        return False

In [ ]:
with DemoOrderTransaction() as transaction:
    transaction.add("create ORD-201")
    transaction.add("reserve inventory")

In [ ]:
try:
    with DemoOrderTransaction() as transaction:
        transaction.add("create ORD-202")
        transaction.add("charge payment")
        raise RuntimeError("payment service rejected the charge")
except RuntimeError as error:
    print("ORDER SERVICE caught:", error)

A real database connection defines its own transaction behavior, so follow that library's documentation. The example above is an in-memory lifecycle demonstration.

## 10. Generator-based context managers

`contextlib.contextmanager` converts a specially structured generator function into a context manager:

- Code before `yield` acts like `__enter__()`.
- The yielded object becomes the value after `as`.
- Code in `finally` after `yield` acts like reliable cleanup in `__exit__()`.

The function must yield exactly once during a normal use.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_order_file(path, mode="r"):
    print(f"SETUP: opening {path}")
    file = Path(path).open(mode, encoding="utf-8")

    try:
        yield file
    finally:
        print(f"CLEANUP: closing {path}")
        file.close()

In [ ]:
with managed_order_file("generator_managed_orders.txt", "w") as output:
    output.write("ORD-301,delivered\n")
    print("BLOCK: file closed?", output.closed)

print("AFTER: file closed?", output.closed)

The `try/finally` is essential. If the `with` block raises an exception, it is injected at the suspended `yield` point; `finally` still runs and closes the file.

In [ ]:
try:
    with managed_order_file("generator_managed_orders.txt", "r") as source:
        print("READ:", source.readline().strip())
        raise LookupError("order lookup failed")
except LookupError as error:
    print("OUTER CODE caught:", error)

print("File closed after exception?", source.closed)

## 11. Timing a block with a context manager

A context manager may manage temporary state rather than a physical resource. This reusable timer measures the duration of any block.

In [ ]:
from time import perf_counter

class Timer:
    def __init__(self, label):
        self.label = label
        self.started_at = None
        self.elapsed = None

    def __enter__(self):
        self.started_at = perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = perf_counter() - self.started_at
        print(f"{self.label} took {self.elapsed:.6f} seconds")
        return False

with Timer("Calculate order total") as timer:
    order_total = sum(range(100_000))

print("Total:", order_total)
print("Recorded elapsed time:", timer.elapsed)

## 12. Multiple context managers

One `with` statement can manage multiple resources. They enter from left to right and exit in reverse order—similar to nested `with` blocks.

In [ ]:
input_path = Path("context_input.txt")
output_path = Path("context_output.txt")
input_path.write_text("ORD-401,packed\nORD-402,shipped\n", encoding="utf-8")

with (
    input_path.open("r", encoding="utf-8") as source,
    output_path.open("w", encoding="utf-8") as destination,
):
    for line in source:
        destination.write(line.upper())

print(output_path.read_text(encoding="utf-8"))

## 13. Objects that are not context managers

An ordinary list, dictionary, or integer does not implement `__enter__()` and `__exit__()`, so it cannot be used directly after `with`.

In [ ]:
try:
    with ["ORD-1", "ORD-2"] as orders:
        print(orders)
except TypeError as error:
    print(type(error).__name__ + ":", error)

`contextlib` contains adapters for common situations. For example, `contextlib.closing(resource)` can call `.close()` on an object that has a close method but does not implement the context-manager protocol itself.

## 14. Synchronous versus asynchronous context managers

This lesson uses synchronous context managers:

```python
with manager:
    ...
```

Asynchronous libraries may expose `__aenter__()` and `__aexit__()` instead. Those objects are consumed with `async with` inside an `async def` function:

```python
async with async_manager as resource:
    ...
```

A synchronous manager and an asynchronous manager use related but separate protocols.

## 15. Context manager versus generator versus iterator

| Concept | Main purpose | Key protocol |
|---|---|---|
| Iterable | Can provide an iterator | `__iter__()` |
| Iterator | Produces successive values | `__iter__()`, `__next__()` |
| Generator | Iterator built with suspension and `yield` | Iterator protocol |
| Context manager | Controls entry and cleanup around a block | `__enter__()`, `__exit__()` |

`contextlib.contextmanager` connects two concepts: it uses a generator internally to implement the context-manager protocol. That does not mean every generator is automatically a context manager.

## 16. Summary

- `with` is Python syntax for using a context manager.
- `__enter__()` prepares the context and supplies the optional `as` value.
- `__exit__()` performs cleanup and receives exception information.
- File objects close automatically because their context manager closes them in `__exit__()`.
- Returning a truthy value from `__exit__()` suppresses the active exception.
- A custom class can implement the protocol directly.
- `@contextmanager` can build the same lifecycle using setup before `yield` and cleanup after it.
- Context managers make resource ownership, cleanup, transaction decisions, and temporary state changes safer and clearer.

## 17. Practice

1. Add a line counter to `OrderFileManager` and print the count during exit.
2. Create a context manager that temporarily changes a dictionary value and restores it afterward.
3. Modify `DemoOrderTransaction` to store a `committed` or `rolled_back` status.
4. Write a generator-based timer using `@contextmanager`.
5. Test your manager with both a successful block and a block that raises an exception.